In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install skl2onnx scikit-learn pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 122.2 MB/s eta 0:00:00


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# 1. Configuración de ruta (Basado en tu imagen)
base_path = '/content/drive/MyDrive/Dataset/'

# 2. Cargar los archivos CSV crudos
df_daily = pd.read_csv(base_path + 'aquatracking.dailyconsumptions.csv')
df_homes = pd.read_csv(base_path + 'aquatracking.homes.csv')
df_sectors = pd.read_csv(base_path + 'aquatracking.sectors.csv')

In [10]:
# 3. Procesamiento y Unión de Datos (Merge)
# Preparamos homes y sectors para unir
df_homes_clean = df_homes[['_id', 'members', 'sectorId']].rename(columns={'_id': 'homeId_ref'})
df_sectors_clean = df_sectors[['_id', 'name']].rename(columns={'_id': 'sectorId', 'name': 'sectorName'})

# Unimos Daily con Homes
merged_df = pd.merge(df_daily, df_homes_clean, left_on='homeId', right_on='homeId_ref', how='left')

# Unimos el resultado con Sectors
final_df = pd.merge(merged_df, df_sectors_clean, on='sectorId', how='left')

# Limpieza de fechas y columnas
final_df['date'] = pd.to_datetime(final_df['date'])
final_df['month'] = final_df['date'].dt.month
final_df['day_of_week'] = final_df['date'].dt.dayofweek

In [11]:
# 4. Preparación para ML
# Convertir sectorName a números
final_df = pd.get_dummies(final_df, columns=['sectorName'], drop_first=True)

# Definir Features y Target
# Features: Habitantes, Mes, Día Semana, Sector
features = ['members', 'month', 'day_of_week'] + [col for col in final_df.columns if 'sectorName_' in col]
target = 'totalLiters'

X = final_df[features].fillna(0) # Rellenar nulos con 0 por seguridad
y = final_df[target].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# 5. Entrenamiento (Random Forest vs Regresión Lineal)
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
mae_lr = mean_absolute_error(y_test, model_lr.predict(X_test))

model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
mae_rf = mean_absolute_error(y_test, model_rf.predict(X_test))

print(f"Error Regresión Lineal: {mae_lr:.2f}")
print(f"Error Random Forest: {mae_rf:.2f}")

best_model = model_rf if mae_rf < mae_lr else model_lr
print(f"Exportando: {type(best_model).__name__}")

Error Regresión Lineal: 51.29
Error Random Forest: 42.61
Exportando: RandomForestRegressor


In [13]:
# 6. Exportar a ONNX
initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_sklearn(best_model, initial_types=initial_type)

with open("modelo_consumo.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Archivo modelo_consumo.onnx generado.")

Archivo modelo_consumo.onnx generado.
